# Lire les logits et choisir une classe avec argmax

Distinguer scores, annotations et décisions sur des axes faciles à reconnaître.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#du-pipeline-au-modele).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Avant de produire des scores avec une couche, apprenons à les lire. Nous choisissons ici les nombres à la main : ils ne viennent ni d’un modèle ni des fichiers audio.

| Ligne et extrait | Voix : classe 0 | Percussion : classe 1 | Annotation attendue |
| --- | --- | --- | --- |
| 0 : Voix A | 1,5 | −1,5 | 0 |
| 1 : Percussion B | 4,5 | −4,5 | 1 |
| 2 : Voix C | 2,5 | −2,5 | 0 |

Les scores ont la forme (3,2). Trois extraits et deux classes rendent les axes différents. Le score voix gagne aussi pour B : cette erreur est volontaire.

Un score brut peut être négatif ou supérieur à un. Il ne s’agit pas encore d’une probabilité.

argmax cherche une position, pas la valeur maximale. dim=1 compare les deux colonnes de chaque ligne : les classes. Dans les trois lignes, la colonne zéro gagne.

## Démonstration 1 · Lire la décision, puis modifier une copie

Observer la correspondance entre une sortie et chaque extrait.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
# Scores choisis à la main ; colonnes : voix 0, percussion 1.


### Calculer l’état suivant

Les deux niveaux de crochets forment des lignes contenant chacune deux scores. cibles garde une annotation par ligne.

In [4]:
scores = torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('scores', scores)
observer('cibles', cibles)

scores forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

1,5 dépasse −1,5 ; 4,5 dépasse −4,5 ; 2,5 dépasse −2,5. Les trois maxima sont en colonne 0.

Avec dim=0, chaque colonne compare les trois extraits. Le résultat répondrait à une autre question : quel extrait gagne pour chaque classe ?

In [6]:
predictions = scores.argmax(dim=1)


In [7]:
observer('predictions', predictions)

predictions forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 0, 0]


### Calculer l’état suivant

== compare élément par élément : 0==0, 0==1, puis 0==0. Chaque booléen reste associé à son extrait.

In [8]:
accords = predictions == cibles


In [9]:
observer('accords', accords)

accords forme (3,) type torch.bool requires_grad False
  valeurs : [True, False, True]


### Lire les sorties

tolist() transforme les valeurs en listes Python pour les afficher. Cette conversion ne réécrit pas les données du tenseur.

In [10]:
print("scores", scores.tolist(), "forme", tuple(scores.shape))
print("predictions", predictions.tolist(), "forme", tuple(predictions.shape))
print("accords", accords.tolist())


scores [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]] forme (3, 2)
predictions [0, 0, 0] forme (3,)
accords [True, False, True]


### Calculer l’état suivant

clone() crée un tenseur avec un stockage distinct. Nous voulons tester une modification sans écraser les scores d’origine.

In [11]:
scores_modifies = scores.clone()


In [12]:
observer('scores_modifies', scores_modifies)

scores_modifies forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


### Calculer l’état suivant

[1,1] signifie deuxième extrait, deuxième classe. Le score 5 dépasse alors 4,5, tandis que les autres lignes restent inchangées.

In [13]:
scores_modifies[1, 1] = 5.0


In [14]:
observer('scores_modifies', scores_modifies)

scores_modifies forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, 5.0], [2.5, -2.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [15]:
print("apres modification", scores_modifies.argmax(dim=1).tolist())
print("original", scores.tolist())

apres modification [0, 1, 0]
original [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


La modification manuelle corrige la décision de B. Nous n’avons encore entraîné aucun paramètre : nous avons seulement vérifié comment une préférence devient une classe.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Prévoyez les indices de classes renvoyés par argmax sur les colonnes.

In [16]:
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_predictions = ...  # Remplacer les points par votre réponse.
if exo_predictions is Ellipsis or exo_predictions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_predictions),exo_scores.argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Les trois lignes ont leur maximum en colonne 0. La taille du lot n’est pas le nombre de classes.

</details>

### Exercice 2

Reliez chaque décision à sa cible : quelles lignes sont correctement classées ?

In [17]:
exo_predictions = torch.tensor([0,0,0])
exo_cibles = torch.tensor([0,1,0])
exo_accords = ...  # Remplacer les points par votre réponse.
if exo_accords is Ellipsis or exo_accords is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_accords),exo_predictions == exo_cibles)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Comparez les deux vecteurs position par position, en conservant l’ordre des extraits.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_predictions = torch.tensor([0,0,0])
if exo_predictions is Ellipsis or exo_predictions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_predictions),exo_scores.argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_predictions = torch.tensor([0,0,0])
exo_cibles = torch.tensor([0,1,0])
exo_accords = torch.tensor([True,False,True])
if exo_accords is Ellipsis or exo_accords is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_accords),exo_predictions == exo_cibles)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()